```
<13_.ipynb>

제미나이 의존도: 10-20%

아직 진행중
```

In [20]:
import json
import os

In [21]:
# JSON 파일 읽어오기 (메모리에 딕셔너리로 로드)
with open('dataset/skyFusion/SkyFusion/train/_annotations.coco.json', 'r') as f:
    data = json.load(f)

# 가장 최상위 key들이 뭐가 있는지 확인해보기
print(f"JSON 최상위 키 목록: {data.keys()}")

# 데이터가 각각 몇 개씩 들어있는지 확인
print(f"총 이미지 수: {len(data['images'])}")
print(f"총 라벨(박스) 수: {len(data['annotations'])}")
print(f"클래스 종류: {data['categories']}\n")


image_dict = {}

images = data['images']
for image in images:
    image_id = image['id']
    file_name = image['file_name']
    height = image['height']
    width = image['width']
    image_dict[image_id] = (file_name, height, width)

# for key, value in list(image_dict.items())[:3]:
    # print(f'{key}: {value}')


# images에는 고유한 id가 있다. 그리고 file_name, height, width를 가지고 있다.
# labels에는 image_id를 가지고 있고, 이것을 images의 id와 연결시켜야 한다.

annotations = data['annotations']

label_save_path = 'dataset/skyFusion/SkyFusion/labels'
os.makedirs(label_save_path, exist_ok=True)

for anno in data['annotations'][:10]:
    img_id = anno['image_id']
    cat_id = anno['category_id'] - 1
    xmin, ymin, box_w, box_h = anno['bbox']

    file_name, img_h, img_w = image_dict[img_id]

    x_center = xmin + (box_w / 2)
    y_center = ymin + (box_h / 2)

    norm_x = x_center / img_w
    norm_y = y_center / img_h
    norm_w = box_w / img_w
    norm_h = box_h / img_h

    pure_name, ext = os.path.splitext(file_name)
    txt_filename = f"{pure_name}.txt"
    save_txt_path = os.path.join(label_save_path, txt_filename)

    line = f"{cat_id} {norm_x:.6f} {norm_y:.6f} {norm_w:.6f} {norm_h:.6f}\n"

    with open(save_txt_path, 'a') as f:
        f.write(line)
print('success')
    
# {'id': 0, 
#  'license': 1, 
#  'file_name': 'P2491__1-0__1200___1764_png_jpg.rf.00342c6c14ae53b3bfadd7995643e1bc.jpg', 
#  'height': 640, 
#  'width': 640, 
#  'date_captured': '2023-12-20T13:55:24+00:00'}

# {'id': 0, 
#  'image_id': 0, 
#  'category_id': 3, 
#  'bbox': [259, 49, 4.8, 9.6], 
#  'area': 46.08, 
#  'segmentation': [[264, 48.8, 259.2, 48.8, 259.2, 58.4, 264, 58.4, 264, 48.8]], 
#  'iscrowd': 0}

JSON 최상위 키 목록: dict_keys(['info', 'licenses', 'categories', 'images', 'annotations'])
총 이미지 수: 2094
총 라벨(박스) 수: 43575
클래스 종류: [{'id': 1, 'name': 'Aircraft'}, {'id': 2, 'name': 'ship'}, {'id': 3, 'name': 'vehicle'}]

success


In [25]:
split_data = ["train", "valid", "test"]
for split_type in split_data:
    count = 0
    label_save_path = f'dataset/skyFusion/SkyFusion/{split_type}_labels'
    os.makedirs(label_save_path, exist_ok=True)
    with open(f'dataset/skyFusion/SkyFusion/{split_type}/_annotations.coco.json') as f:
        data = json.load(f)

    image_dict = {}
    for image in data['images']:
        image_id = image['id']
        image_dict[image_id] = (image['file_name'], image['height'], image['width'])

    annotations = data['annotations']
    for anno in annotations:
        image_id = anno['image_id']
        file_name, img_height, img_width = image_dict[image_id]

        category_id = anno['category_id'] - 1

        xmin, ymin, box_w, box_h = anno['bbox']

        center_x = xmin + (box_w / 2)
        center_y = ymin + (box_h / 2)

        norm_x = center_x / img_width
        norm_y = center_y / img_height
        norm_w = box_w / img_width
        norm_h = box_h / img_height

        line = f'{category_id} {norm_x:.6f} {norm_y:.6f} {norm_w:.6f} {norm_h:.6f}\n'
        pure_name, ext = os.path.splitext(file_name)
        save_txt_path = os.path.join(label_save_path, f'{pure_name}.txt')
        with open(save_txt_path, 'a') as f:
            f.write(line)

        count += 1
    
    print(f'{split_type} count: {count}')
print('success')

train count: 43575
valid count: 8387
test count: 11751
success
